# 01 · Cadrage métier et analyse des données (axes 1, 2, 3)
**Décision** : pour chaque demande de prêt personnel, accorder ou refuser (aide à la décision de l'analyste crédit).
**Cible** : `loan_status` (1 = défaut, 0 = remboursé), connue pour tout l'historique → classification binaire supervisée.
**Situation actuelle (hypothèse)** : refus des notes internes D à G. **Critère de succès** : Recall ≥ 80 % avec Precision ≥ 50 %, et coût d'erreur réduit d'au moins 30 % vs la règle actuelle.
Détails complets dans le rapport (axes 1 et 2).

In [1]:
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, precision_recall_curve,
                             roc_auc_score)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, cross_val_predict,
                                     cross_val_score, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

sys.path.insert(0, os.path.abspath("../src"))
from features import CAT_COLS, NUM_COLS, TARGET, add_features, clean, load_raw
from costs import confusion, metrics, roi, total_cost

warnings.filterwarnings("ignore")
RS = 42
ROOT = os.path.abspath("..")
FIG = os.path.join(ROOT, "reports", "figures")
os.makedirs(FIG, exist_ok=True)

# ---------------- hypothèses business (axes 6-7), toutes testées en sensibilité -------------
LGD, MARGIN = 0.60, 0.12          # perte si défaut = 60 % du montant ; marge nette perdue si refus = 12 %
N_ANNUAL = 32_000                 # nombre de demandes traitées par an (hypothèse = taille du jeu)
COUT_PROJET = 150_000             # 1re année : développement 80 k + intégration/validation 30 k + exploitation 40 k
GAIN_REALISE = 0.70               # décote : seule une part du gain mesuré hors échantillon est réalisée en production
GUARDRAIL_PRECISION = 0.50        # au plus 1 client sain refusé pour 1 défaut évité

NAVY, TEAL, ORANGE, GREY, RED = "#1F3A5F", "#2A9D8F", "#E76F51", "#8D99AE", "#C0392B"
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 10, "axes.titleweight": "bold", "figure.dpi": 150})
R = {}                            # tous les résultats
t0 = time.time()

def save(fig, name):
    fig.savefig(os.path.join(FIG, name), bbox_inches="tight"); plt.close(fig)

## Chargement, nettoyage, variables construites, split
Le nettoyage (doublons, valeurs impossibles) est déterministe : il ne dépend d'aucune statistique apprise, donc sans fuite. Le split est fait **avant** toute préparation apprise.

In [2]:
# ------------------------------------------------------------------ 1. données
raw = load_raw(os.path.join(ROOT, "data", "credit_risk_dataset.csv"))
R["raw"] = {"n": len(raw), "cols": raw.shape[1],
            "missing": raw.isna().sum()[lambda s: s > 0].to_dict(),
            "age_max": float(raw.person_age.max()), "emp_max": float(raw.person_emp_length.max()),
            "income_max": float(raw.person_income.max())}
df, qc = clean(raw)
R["clean"] = qc
df = add_features(df)
R["default_rate"] = float(df[TARGET].mean())
R["n_clean"] = len(df)
R["rate_by"] = {
    "grade": df.groupby("loan_grade")[TARGET].mean().round(3).to_dict(),
    "prior_default": df.groupby("cb_person_default_on_file")[TARGET].mean().round(3).to_dict(),
    "home": df.groupby("person_home_ownership")[TARGET].mean().round(3).to_dict(),
    "intent": df.groupby("loan_intent")[TARGET].mean().round(3).to_dict(),
}
df["lpi_bin"] = pd.qcut(df["loan_percent_income"], 8, duplicates="drop")
R["rate_by"]["lpi_bin_first_last"] = [float(df.groupby("lpi_bin", observed=True)[TARGET].mean().iloc[0]),
                                      float(df.groupby("lpi_bin", observed=True)[TARGET].mean().iloc[-1])]
R["missing_rate_by_default"] = {
    "int_rate_missing": df.assign(m=df.loan_int_rate.isna()).groupby("m")[TARGET].mean().round(3).to_dict()}

# ------------------------------------------------------------------ 2. split (AVANT toute préparation)
X = df[NUM_COLS + CAT_COLS]
y = df[TARGET]
idx_tr, idx_te = train_test_split(df.index, test_size=0.2, stratify=y, random_state=RS)
X_tr, X_te, y_tr, y_te = X.loc[idx_tr], X.loc[idx_te], y.loc[idx_tr], y.loc[idx_te]
amt_tr, amt_te = df.loc[idx_tr, "loan_amnt"].values, df.loc[idx_te, "loan_amnt"].values
R["split"] = {"train": len(X_tr), "test": len(X_te), "default_train": float(y_tr.mean()),
              "default_test": float(y_te.mean())}


def make_prep(num_cols, cat_cols):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                          ("sc", StandardScaler())]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])


def make_pipe(clf, num_cols=NUM_COLS, cat_cols=CAT_COLS):
    return Pipeline([("prep", make_prep(num_cols, cat_cols)), ("clf", clf)])


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RS)

In [3]:
print(R["raw"]); print(R["clean"]); print("taux de défaut :", round(R["default_rate"], 3)); print(R["split"])

{'n': 32580, 'cols': 12, 'missing': {'person_emp_length': 895, 'loan_int_rate': 3116}, 'age_max': 144.0, 'emp_max': 123.0, 'income_max': 2039784.0}
{'lignes_brutes': 32580, 'doublons': 165, 'valeurs_impossibles': 6, 'lignes_finales': 32409}
taux de défaut : 0.219
{'train': 25927, 'test': 6482, 'default_train': 0.21869093994677363, 'default_test': 0.218759642085776}


## Audit de fuite
Aucune colonne ne décrit l'après-octroi. `loan_grade` et `loan_int_rate` sont fixées par la banque à l'octroi (connues à la décision mais circulaires) : leur poids est mesuré par ablation dans le notebook 03.

## Graphiques (chaque figure est interprétée dans le rapport)

In [4]:
# fig 2 : EDA
fig, axs = plt.subplots(2, 2, figsize=(8.6, 6))
ax = axs[0, 0]
vc = df[TARGET].value_counts(normalize=True).sort_index()
ax.bar(["Remboursé (0)", "Défaut (1)"], vc.values * 100, color=[TEAL, ORANGE])
for i, v in enumerate(vc.values): ax.text(i, v * 100 + 1, f"{v*100:.1f} %", ha="center", fontweight="bold")
ax.set_ylabel("% des demandes"); ax.set_title("Classe défaut minoritaire (~22 %)"); ax.set_ylim(0, 95)
ax = axs[0, 1]
g = df.groupby("loan_grade")[TARGET].mean() * 100
ax.bar(g.index, g.values, color=[TEAL if v < 25 else ORANGE for v in g.values])
ax.set_ylabel("taux de défaut (%)"); ax.set_xlabel("note interne (A = peu risqué)")
ax.set_title("Le défaut explose à partir de la note D")
ax = axs[1, 0]
g = df.groupby("lpi_bin", observed=True)[TARGET].mean() * 100
ax.plot(range(len(g)), g.values, marker="o", color=NAVY)
ax.set_xticks(range(len(g))); ax.set_xticklabels([f"{b.mid:.2f}" for b in g.index], fontsize=7)
ax.set_xlabel("charge de la dette : montant / revenu (centre de la tranche)"); ax.set_ylabel("taux de défaut (%)")
ax.set_title("Plus la charge pèse sur le revenu, plus le risque monte")
ax = axs[1, 1]
cat = pd.concat([df.groupby("cb_person_default_on_file")[TARGET].mean(),
                 df.groupby("person_home_ownership")[TARGET].mean().loc[["OWN", "MORTGAGE", "RENT"]]]) * 100
lab = ["Pas de défaut\npassé", "Défaut\npassé", "Proprié-\ntaire", "Hypo-\nthèque", "Locataire"]
ax.bar(lab, cat.values, color=[TEAL, ORANGE, TEAL, GREY, ORANGE]); ax.tick_params(axis="x", labelsize=7)
ax.set_ylabel("taux de défaut (%)"); ax.set_title("Historique de crédit et statut de logement")
fig.tight_layout(); save(fig, "fig2_eda.png")

In [5]:
# fig 3 : corrélations
num_eda = df[["person_age", "person_income", "person_emp_length", "loan_amnt", "loan_int_rate",
              "loan_percent_income", "cb_person_cred_hist_length", "grade_num", TARGET]]
cm = num_eda.corr(method="spearman")
lbl = ["âge", "revenu", "ancienneté\nemploi", "montant", "taux\nintérêt", "montant/\nrevenu",
       "hist.\ncrédit", "note\ninterne", "DÉFAUT"]
fig, ax = plt.subplots(figsize=(6.4, 5))
im = ax.imshow(cm.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(9)); ax.set_xticklabels(lbl, fontsize=6.5); ax.set_yticks(range(9)); ax.set_yticklabels(lbl, fontsize=6.5)
for i in range(9):
    for j in range(9): ax.text(j, i, f"{cm.values[i, j]:.2f}", ha="center", va="center", fontsize=6)
ax.set_title("Corrélations (Spearman) : le défaut suit note, taux et charge de la dette")
fig.colorbar(im, shrink=.8); save(fig, "fig3_corr.png")